In [1]:
#%% Imports and settings
import sciris as sc
import starsim as ss
import numpy as np
import pandas as pd
import matplotlib.dates as mdates
from enum import IntEnum, auto
import seaborn as sns
from pathlib import Path
import pickle
from collections import defaultdict

n_agents = 20000
debug = False # If true, will run in serial

/Users/nparke19/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def starsim_sim_function(district_file, location_data_path_name): 
    
    ### first, define all arguments
    district_file = district_file
    location_data_path_name = location_data_path_name

    ### create LocationDemographics class to allow for dynamically updating location data
    class LocationDemographics(ss.Demographics):
        """ Track and update each person's location at every timestep """
        def __init__(self, location_dataset, locations=None, pars=None, **kwargs):
            """
            Initialize locations and movement probabilities.

            Args:
                locations (list): List of possible locations.
                movement_probs (dict): Dictionary defining movement probabilities between locations.
            """
            super().__init__()
            
            self.define_pars(
                locations=locations , 
                location_dataset=location_dataset
                
            )
            self.update_pars(pars, **kwargs)
            
            return
        
        def init_pre(self, sim):
            """ 
            Initialize the location for each person at the start of the simulation 
            """
            super().init_pre(sim)
            people = self.sim.people
            loc_data = self.pars.location_dataset
            print("Number of people:", len(self.sim.people))


            people.location = loc_data[(loc_data["step"] == 0)]["location"].values
            print("peoples.location:", people.location)
            
            return

            
        
        def step(self):
            """ Update each person's location at every timestep """
            sim = self.sim
            people = self.sim.people
            loc_data = self.pars.location_dataset

            if self.ti % 5 == 0:
                print("time step", self.ti)

            
            current_step_data = loc_data[loc_data["step"] == self.ti][["uid", "location"]]


            location_array = current_step_data["location"].values
            people.location[:] = location_array
            
            return

        def init_results(self):
            """ Initialize tracking of locations over time """
            super().init_results()
            for loc in self.pars.locations:
                self.define_results(ss.Result(f'num_in_{loc}', dtype=int, label=f'People in {loc}'))
            return
        
        def update_results(self):
            """ Store location data at each timestep """
            people = self.sim.people
            for loc in self.pars.locations:
                self.results[f'num_in_{loc}'][self.ti] = np.sum(people.location == loc)
            return


    ### load in the district data and set the LOCATION variable, which assigns a unqiue number to each location 
    districts = pd.read_csv(district_file)

    district_names = districts["District"].tolist()

    LOCATION = IntEnum("LOCATION", {name.upper().replace(" ", "_"): i for i, name in enumerate(district_names)})


    ### create a CustomNet class which defines the network structure manually
    class CustomNet(ss.Network):
        def __init__(self, contact_dict):
            super().__init__()
            self.contact_dict = contact_dict

        def step(self):
            contacts = self.contact_dict[self.ti]
            self.edges.p1 = contacts['p1']
            self.edges.p2 = contacts['p2']
            self.edges.beta = np.ones(len(self))
            self.validate()


    ### create SEIR class, which codes in the structure and dynamics of our your disease.
    ### This needs duration of exposure (distribution) argument. 
    class SEIR(ss.SIR):
        def __init__(self, pars=None, *args, **kwargs):
            super().__init__()
            self.define_pars(dur_exp = ss.lognorm_ex(mean = 1.6, std  = 0.3)
            )
            self.update_pars(pars, **kwargs)

            # Additional states beyond the SIR ones
            self.define_states(
                ss.State('exposed', label='Exposed'),
                ss.FloatArr('ti_exposed', label='TIme of exposure'),
            )
            return

        @property
        def infectious(self):
            return self.infected 

        def step_state(self):
            """ Make all the updates from the SIR model """
            # Perform SIR updates
            super().step_state()

            # Additional updates: progress exposed -> infected
            infected = self.exposed & (self.ti_infected <= self.ti)
            self.exposed[infected] = False
            self.infected[infected] = True
            return

        def step_die(self, uids):
            super().step_die(uids)
            self.exposed[uids] = False
            return

        def set_prognoses(self, uids, sources=None):
            """ Carry out state changes associated with infection """
            super().set_prognoses(uids, sources)
            ti = self.ti
            self.susceptible[uids] = False
            self.exposed[uids] = True
            self.ti_exposed[uids] = ti

            # Calculate and schedule future outcomes
            dur_exp = self.pars['dur_exp'].rvs(uids)
            self.ti_infected[uids] = ti + dur_exp
            dur_inf = self.pars['dur_inf'].rvs(uids)
            will_die = self.pars['p_death'].rvs(uids)
            self.ti_recovered[uids[~will_die]] = ti + dur_inf[~will_die]
            self.ti_dead[uids[will_die]] = ti + dur_inf[will_die]
            return

        def plot(self):
            """ Update the plot with the exposed compartment """
            with ss.options.context(jupyter=False):
                fig = super().plot()
                ax = plt.gca()
                res = self.results.n_exposed
                ax.plot(res.timevec, res, label=res.label)
                plt.legend()
            return ss.return_fig(fig)
        
    LOCATION = IntEnum("LOCATION", {name.upper().replace(" ", "_"): i for i, name in enumerate(district_names)})
    LOCATIONS = list(LOCATION)

    ### set the parameters of the simulation. The length of the simulation is adjustable.     
    pars = sc.objdict(n_agents = 20000, start = sc.date('2020-9-17'),
        stop = sc.date('2021-03-01'),
        dt = 1,
        unit = 'day',)

    ### Define the simulate_function, which puts everything together and runs the simulation
    ### Needs multiple arguments: file_path of the location data,  
    def simulate_function(location_data_path):
        
        #load in the dataset and transform locations into numbers. 
        location_sim= pd.read_csv(location_data_path)
        location_sim = pd.DataFrame(location_sim)
        location_sim["location"] = location_sim["location"].apply(lambda x: int(LOCATION[x.upper().replace(" ", "_")]))
        location_sim = location_sim[location_sim["step"] <= 225]

        # Create the People object
        location_arr= ss.FloatArr("location", default=location_sim[location_sim["step"] == 0]["location"].values)
        ppl_return = ss.People(n_agents=20000, extra_states=location_arr)

        
        def seeding_return(self, sim, uids):
            p = np.zeros(len(uids))
            # total number of infections
            n_total = int(round(0.0007 * len(uids)))
            # group uids by location
            loc_groups = location_sim.groupby("location")["uid"].apply(list)
            seeded_uids = []
            # 1 infection per location
            for loc, loc_uids in loc_groups.items():
                uid = np.random.choice(loc_uids)
                seeded_uids.append(uid)
            # how many left to allocate after guaranteeing 1 per location
            n_remaining = max(n_total - len(seeded_uids), 0)
            # choose remaining infections at random from rest of population
            remaining_pool = list(set(uids) - set(seeded_uids))
            if n_remaining > 0:
                seeded_uids.extend(np.random.choice(remaining_pool, size=n_remaining, replace=False))
            # set p=1 for seeded uids
            p[np.isin(uids, seeded_uids)] = 1
            return p
        
        
        seir_disease_return = SEIR(init_prev = ss.bernoulli(p = seeding_return),
                                    beta = ss.beta(0.077),
                                    dur_inf = ss.poisson(7), 
                                    p_death = 0)

        # create the contact network
        # Parameters
        from collections import defaultdict

        lambda_contacts = 5.03  # Mean number of contacts per person per day
        contacts_dict = {}

        # Group once by step and location
        grouped = location_sim.groupby(["step", "location"])

        for (step, location), group in grouped:
            agents = group["uid"].values
            n = len(agents)

            if n <= 1:
                continue

            # Estimate total number of edges: n * mean_contacts / 2
            expected_edges = int(n * lambda_contacts / 2)
            
            # Sample contact pairs
            idx1 = np.random.choice(n, size=expected_edges, replace=True)
            idx2 = np.random.choice(n, size=expected_edges, replace=True)

            # Remove self-contacts
            mask = idx1 != idx2
            pairs = np.stack([agents[idx1[mask]], agents[idx2[mask]]], axis=1)

            # Sort each pair to avoid (a,b) vs (b,a)
            pairs = np.sort(pairs, axis=1)

            # Remove duplicate pairs
            pairs = np.unique(pairs, axis=0)

            # Initialize the step entry if not yet added
            if step not in contacts_dict:
                contacts_dict[step] = {"p1": [], "p2": []}
            
            contacts_dict[step]["p1"].extend(pairs[:, 0])
            contacts_dict[step]["p2"].extend(pairs[:, 1])

        for step in contacts_dict:
            contacts_dict[step]["p1"] = np.array(contacts_dict[step]["p1"])
            contacts_dict[step]["p2"] = np.array(contacts_dict[step]["p2"])
        

        ### create the custom network based on the contacts dictionary
        custom_network_return = CustomNet(contacts_dict)

        ### make a list of all locations used in the travel data
        LOCATIONS = list(LOCATION)

        ### initialize the location demographic class and the analyzer
        demographics_sim = [LocationDemographics(location_dataset= location_sim, locations = LOCATIONS)]


        ### run the sim
        simulation = ss.Sim(pars, diseases = seir_disease_return, people= ppl_return, networks=custom_network_return, demographics = demographics_sim)
        
        return simulation

    final_sim = simulate_function(location_data_path_name)
                      
    return final_sim

In [3]:
def make_sim():
    sim = starsim_sim_function(district_file = "/Users/nparke19/Documents/Mobility/Zambia/Cleaning_modeling_code/sim_objects/Penn_districts.csv",
                     location_data_path_name= "/Users/nparke19/Documents/Mobility/Alpha_wave/Location_sims/Penn_no_mobility/sim_no_mob_1.csv")

    return sim



In [4]:
# Define the calibration parameters
calib_pars = dict(
    beta = dict(low=0.01, high=0.15, guess=0.04, suggest_type='suggest_float')) # Default type is suggest_float, no need to re-specify)

In [5]:
def build_sim(sim, calib_pars, **kwargs):
    """ Modify the base simulation by applying calib_pars """

    reps = kwargs.get('n_reps', 1)

    seir = sim.pars.diseases # There is only one disease in this simulation and it is a SIR


    for k, pars in calib_pars.items():
        if k == 'rand_seed':
            sim.pars.rand_seed = pars
            continue

        v = pars['value']
        if k == 'beta':
            seir.pars.beta = ss.beta(v)
        else:
            raise NotImplementedError(f'Parameter {k} not recognized')

    if reps == 1:
        return sim

    # Ignoring the random seed if provided via the reseed=True option in Calibration
    ms = ss.MultiSim(sim, iterpars=dict(rand_seed=np.random.randint(0, 1e6, reps)), initialize=True, debug=True, parallel=False)
    return ms

In [6]:
def eval(sim, expected):
    """
    Evaluate the simulation against multiple expected (date, value) pairs.
    
    Args:
        sim (Sim or MultiSim): The simulation object to evaluate.
        expected (list of tuples): List of (date, prevalence) pairs to compare against.
        
    Returns:
        float: The total squared error across all dates and sims.
    """
    if not isinstance(sim, ss.MultiSim):
        sim = ss.MultiSim(sims=[sim])

    total_error = 0.0
    for s in sim.sims:
        # Check for extinction (all new infections very low or zero)
        if np.all(s.results.seir.new_infections < 10):
            return 1e8  # Immediately return large penalty

        for date, expected_inc in expected:
            # Find index of the given date in the simulation
            ind = np.searchsorted(s.results.timevec, date, side='left')
            if ind >= len(s.results.timevec):
                ind = -1  # Use last available date if overshooting

            sim_inc = s.results.seir.new_infections[ind]
            total_error += (sim_inc - expected_inc) ** 2

    return total_error

In [7]:
# Convert to list of (ss.date, value) tuples

penn_inc_rates = pd.read_csv("~/Desktop/penn_inc_rates.csv")

# Convert to list of (ss.date, value) tuples
penn_inc_rates_df = [(ss.date(row['time_value']), row['cases']) for _, row in penn_inc_rates.iterrows()]
penn_inc_rates_df

[(<2020.10.01>, 7.6450465),
 (<2020.10.02>, 7.794796),
 (<2020.10.03>, 8.1077054),
 (<2020.10.04>, 8.1870503),
 (<2020.10.05>, 8.3155666),
 (<2020.10.06>, 7.828322),
 (<2020.10.07>, 8.3703257),
 (<2020.10.08>, 8.6564143),
 (<2020.10.09>, 8.9693237),
 (<2020.10.10>, 9.5705567),
 (<2020.10.11>, 9.7840056),
 (<2020.10.12>, 10.130441),
 (<2020.10.13>, 10.5148725),
 (<2020.10.14>, 10.4243523),
 (<2020.10.15>, 10.7394967),
 (<2020.10.16>, 10.9920593),
 (<2020.10.17>, 11.1663945),
 (<2020.10.18>, 11.2893232),
 (<2020.10.19>, 11.4379551),
 (<2020.10.20>, 11.6905177),
 (<2020.10.21>, 11.9464329),
 (<2020.10.22>, 12.501847),
 (<2020.10.23>, 13.0807293),
 (<2020.10.24>, 13.3768757),
 (<2020.10.25>, 13.9009989),
 (<2020.10.26>, 14.387126),
 (<2020.10.27>, 15.3124437),
 (<2020.10.28>, 16.1606516),
 (<2020.10.29>, 16.5327903),
 (<2020.10.30>, 16.9161042),
 (<2020.10.31>, 17.4368748),
 (<2020.11.01>, 17.807896),
 (<2020.11.02>, 18.1409209),
 (<2020.11.03>, 18.7969131),
 (<2020.11.04>, 19.3400343),
 (

In [8]:

sc.heading('Beginning calibration')

# Make the sim and data
sim = make_sim()

# Make the calibration
calib = ss.Calibration(
    calib_pars = calib_pars,
    sim = sim,
    build_fn = build_sim,
    build_kw = dict(n_reps=1), # Two reps per point
    reseed = True,
    eval_fn = eval, # Will call my_function(msim, eval_kwargs)
    eval_kw = dict(expected=penn_inc_rates_df), # Will call eval(sim, **eval_kw)
    total_trials = 70,
    n_workers = 1, # None indicates to use all available CPUs
    die = True,
    debug = debug,
)

# Perform the calibration
sc.printcyan('\nPeforming calibration...')
calib.calibrate()

# Check
calib.check_fit()



—————————————————————
Beginning calibration
—————————————————————


Peforming calibration...
Removed existing calibration file starsim_calibration.db
sqlite:///starsim_calibration.db


[I 2025-08-21 15:48:32,051] A new study created in RDB with name: starsim_calibration


Initializing sim with 20000 agents
Number of people: 20000
peoples.location: [ 0  0  0 ... 66 66 66]
  Running 2020.09.17 ( 0/166) (0.00 s)  ———————————————————— 1%
time step 0
time step 5
  Running 2020.09.27 (10/166) (0.06 s)  •——————————————————— 7%
time step 10
time step 15
  Running 2020.10.07 (20/166) (0.11 s)  ••—————————————————— 13%
time step 20
time step 25
  Running 2020.10.17 (30/166) (0.15 s)  •••————————————————— 19%
time step 30
time step 35
  Running 2020.10.27 (40/166) (0.20 s)  ••••———————————————— 25%
time step 40
time step 45
  Running 2020.11.06 (50/166) (0.25 s)  ••••••—————————————— 31%
time step 50
time step 55
  Running 2020.11.16 (60/166) (0.30 s)  •••••••————————————— 37%
time step 60
time step 65
  Running 2020.11.26 (70/166) (0.34 s)  ••••••••———————————— 43%
time step 70
time step 75
  Running 2020.12.06 (80/166) (0.38 s)  •••••••••——————————— 49%
time step 80
time step 85
  Running 2020.12.16 (90/166) (0.43 s)  ••••••••••—————————— 55%
time step 90
time s

[I 2025-08-21 15:48:33,475] Trial 0 finished with value: 315037.15962820355 and parameters: {'beta': 0.13308147130925363, 'rand_seed': 919424}. Best is trial 0 with value: 315037.15962820355.


time step 135
  Running 2021.02.04 (140/166) (0.67 s)  ••••••••••••••••———— 85%
time step 140
time step 145
  Running 2021.02.14 (150/166) (0.72 s)  ••••••••••••••••••—— 91%
time step 150
time step 155
  Running 2021.02.24 (160/166) (0.78 s)  •••••••••••••••••••— 97%
time step 160
time step 165
Initializing sim with 20000 agents
Number of people: 20000
peoples.location: [ 0  0  0 ... 66 66 66]
  Running 2020.09.17 ( 0/166) (0.00 s)  ———————————————————— 1%
time step 0
time step 5
  Running 2020.09.27 (10/166) (0.05 s)  •——————————————————— 7%
time step 10
time step 15
  Running 2020.10.07 (20/166) (0.10 s)  ••—————————————————— 13%
time step 20
time step 25
  Running 2020.10.17 (30/166) (0.15 s)  •••————————————————— 19%
time step 30
time step 35
  Running 2020.10.27 (40/166) (0.19 s)  ••••———————————————— 25%
time step 40
time step 45
  Running 2020.11.06 (50/166) (0.24 s)  ••••••—————————————— 31%
time step 50
time step 55
  Running 2020.11.16 (60/166) (0.29 s)  •••••••————————————— 

...

time step 145
  Running 2021.02.14 (150/166) (0.66 s)  ••••••••••••••••••—— 91%
time step 150
time step 155
  Running 2021.02.24 (160/166) (0.71 s)  •••••••••••••••••••— 97%
time step 160
time step 165
Initializing sim with 20000 agents
Number of people: 20000
peoples.location: [ 0  0  0 ... 66 66 66]
  Running 2020.09.17 ( 0/166) (0.00 s)  ———————————————————— 1%
time step 0
time step 5
  Running 2020.09.27 (10/166) (0.05 s)  •——————————————————— 7%
time step 10
time step 15
  Running 2020.10.07 (20/166) (0.10 s)  ••—————————————————— 13%
time step 20
time step 25
  Running 2020.10.17 (30/166) (0.14 s)  •••————————————————— 19%
time step 30
time step 35
  Running 2020.10.27 (40/166) (0.19 s)  ••••———————————————— 25%
time step 40
time step 45
  Running 2020.11.06 (50/166) (0.23 s)  ••••••—————————————— 31%
time step 50
time step 55
  Running 2020.11.16 (60/166) (0.28 s)  •••••••————————————— 37%
time step 60
time step 65
  Running 2020.11.26 (70/166) (0.32 s)  ••••••••———————————— 43%

[I 2025-08-21 15:49:56,565] Trial 69 finished with value: 315537.46098200354 and parameters: {'beta': 0.020370237378321686, 'rand_seed': 506869}. Best is trial 31 with value: 314617.02280040353.


time step 145
  Running 2021.02.14 (150/166) (0.67 s)  ••••••••••••••••••—— 91%
time step 150
time step 155
  Running 2021.02.24 (160/166) (0.72 s)  •••••••••••••••••••— 97%
time step 160
time step 165
Making results structure...
Processed 70 trials; 0 failed
Best pars: {'beta': 0.03995637754676849, 'rand_seed': 493556}
Removed existing calibration file starsim_calibration.db

Checking fit...


/Users/nparke19/Library/Python/3.9/lib/python/site-packages/dill/_dill.py:422: PicklingWarning: Cannot locate reference to <enum 'LOCATION'>.
  StockPickler.save(self, obj, save_persistent_id)
/Users/nparke19/Library/Python/3.9/lib/python/site-packages/dill/_dill.py:422: PicklingWarning: Cannot pickle <enum 'LOCATION'>: __main__.LOCATION has recursive self-references that trigger a RecursionError.
  StockPickler.save(self, obj, save_persistent_id)


PicklingError: Can't pickle <enum 'LOCATION'>: it's not found as __main__.LOCATION